## Pokemon image generation
### Goal - Train a stable diffusion model to *learn* what Pokemon

In [ ]:
# %%
# Prompt Builder + Sampler for Pokémon TI tokens (works with your existing notebook)
# - Build rich prompts from per-Pokémon descriptors + style presets
# - Inject TI vectors for selected token(s)
# - Optional: merge global LoRA once, then sample

from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Optional
import json
import torch
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
from peft import PeftModel
import random
import numpy as np
from PIL import Image

# ---- Config
MODEL_ID = "runwayml/stable-diffusion-v1-5"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
ADAPTER_DIR: Optional[Path] = Path("./results/pokemon-images/lora_peft_minimal")  # set to None to skip global LoRA
TOKEN_MAP_PATH = Path("./datasets/pokemon-images/_ti_tokens.json")
VECTORS_DIR = Path("./results/pokemon-images/ti_embeddings")
OUT_DIR = Path("./results/pokemon-images/samples"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# Whether to retrain token embeddings for all pokemon
OVERWRITE_EXISTING_TOKENS = False

# List of different styles to render the pokemon image
STYLE_PRESETS: Dict[str, Dict[str, str]] = {
    "3d": {
        "positive": "3d render, studio lighting, high detail, global illumination, subsurface scattering",
        "negative": "low quality, blurry, deformed, extra limbs, worst quality, jpeg artifacts",
    },
    "watercolor": {
        "positive": "watercolor painting, textured paper, soft brush strokes, high detail",
        "negative": "cartoonish outline, heavy posterization, low-res, artifacts",
    },
    "card_art": {
        "positive": "illustration, fantasy trading card art, dramatic lighting, intricate details, professional artstation style",
        "negative": "amateur, messy, low detail, anatomy errors, watermark, signature",
    },
}

# To 
COLOR_WORDS = {
    "black","white","gray","grey","brown","cream","beige","ivory","tan",
    "red","maroon","crimson","orange","amber","gold","yellow",
    "green","blue","navy","teal","cyan","turquoise","purple","violet","magenta","pink",
    "blue-green","dark blue-green","light blue","dark green","dark blue"
}
BODYPLAN_WORDS = {
    "biped","bipedal","quadruped","quadrupedal","serpentine","avian","insectoid",
    "mammal","mammalian","feline","canine","ursine","humanoid","draconic","amphibious",
}
ANATOMY_WORDS = {
    "head","face","belly","back","chest","neck","muzzle","snout","jaw","ear","ears","eye","eyes",
    "horn","horns","crest","plume","mane","frill","fin","fins","wing","wings","feather","tail","claw","claws",
    "spike","spikes","spines","quill","quills","shell","carapace","scales","fur","coat","pelt","whisker","teeth","fangs","tusk"
}
ICONIC_EXTRAS = {
    # things to go in "extras" (short fragments)
    "crest","plume","mane","frill","fins","wings","feathers","horns","spikes","spines","quills",
    "shell","carapace","scales","pattern","stripes","spots","banded"
}


### Build descriptors for each pokemon as a JSON file

In [14]:
# %%
# Auto-fill POKEDESC from dataset folders
# - Scans concept folders under CONCEPTS_ROOT
# - Extracts dominant colors from a few images per concept (palette quantization)
# - Builds lightweight descriptors (body/mood/extras templates) you can tweak later
# - Supports optional overrides via JSON (to refine specific entries)

# Inputs 
CONCEPTS_ROOT = Path("./datasets/pokemon-images")
OVERRIDES_JSON: Optional[Path] =  Path("./datasets/pokemon-images/pokedesc_overrides.json")
OUT_JSON = Path("./results/pokemon-images/pokedesc_autofill.json")

# Utilities
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

# basic color names for nearest-neighbor mapping (RGB 0-255)
NAMED_COLORS = {
    "black": (0, 0, 0),
    "white": (255, 255, 255),
    "gray": (127, 127, 127),
    "red": (200, 50, 50),
    "orange": (230, 120, 30),
    "yellow": (240, 210, 60),
    "green": (50, 160, 80),
    "teal": (60, 170, 170),
    "blue": (60, 120, 220),
    "purple": (140, 70, 200),
    "pink": (230, 120, 200),
    "brown": (110, 70, 40),
}

def rgb_to_name(rgb: tuple[int,int,int]) -> str:
    arr = np.array(rgb, dtype=np.int16)
    best, best_name = 10**9, "unknown"
    for name, ref in NAMED_COLORS.items():
        d = np.linalg.norm(arr - np.array(ref, dtype=np.int16))
        if d < best:
            best, best_name = d, name
    return best_name

def dominant_colors(img: Image.Image, k: int = 5, top_n: int = 2) -> List[str]:
    # palette quantization via Pillow (fast & dependency-free)
    small = img.convert("RGB").resize((128, 128), Image.BICUBIC)
    pal = small.convert("P", palette=Image.ADAPTIVE, colors=k)
    pal_rgb = pal.convert("RGB")
    # count colors
    colors = pal.getcolors(128*128) or []  # list of (count, index)
    colors.sort(reverse=True, key=lambda x: x[0])
    names: List[str] = []
    for count, _ in colors[:top_n*2]:
        # sample a pixel with this color index
        # (convert back to RGB image and read pixel from center-ish)
        # approximate by reading palette image’s histogram order
        # simpler: pick median pixel from pal_rgb
        pass
    # Fallback simple approach: sample palette image pixels and count nearest named colors
    arr = np.array(pal_rgb)
    flat = arr.reshape(-1, 3)
    # random sample to speed up
    if flat.shape[0] > 10_000:
        idx = np.random.choice(flat.shape[0], 10_000, replace=False)
        flat = flat[idx]
    # map to names
    name_counts: Dict[str, int] = {}
    for r, g, b in flat:
        n = rgb_to_name((int(r), int(g), int(b)))
        name_counts[n] = name_counts.get(n, 0) + 1
    top = sorted(name_counts.items(), key=lambda x: x[1], reverse=True)
    names = [n for n, _ in top if n != "unknown"][:top_n]
    if not names:
        names = ["gray"]
    return names

TEMPLATE_BODIES = [
    "compact creature, stylized proportions",
    "creature with simplified anatomy, iconic silhouette",
    "fantasy mascot-like creature",
]
TEMPLATE_MOODS = [
    "cheerful", "calm", "vigilant", "mischievous", "stoic", "energetic"
]
TEMPLATE_EXTRAS = [
    "clean outline, subtle rim light",
    "dynamic pose, slight motion blur",
    "soft bounce light, gentle shadow on ground",
]


def build_pokedesc(concepts_root: Path, max_imgs_per_concept: int = 4) -> Dict[str, Dict[str, str]]:
    rng = random.Random(42)
    desc: Dict[str, Dict[str, str]] = {}
    for folder in sorted(p for p in concepts_root.iterdir() if p.is_dir()):
        name = folder.name
        imgs = [p for p in folder.rglob("*") if p.suffix.lower() in IMG_EXT]
        if not imgs:
            continue
        rng.shuffle(imgs)
        picks = imgs[:max_imgs_per_concept]

        colors_all: List[str] = []
        for p in picks:
            try:
                with Image.open(p) as im:
                    colors_all += dominant_colors(im, k=6, top_n=2)
            except Exception:
                continue
        # consolidate top 2 color names
        color_counts: Dict[str,int] = {}
        for c in colors_all:
            color_counts[c] = color_counts.get(c, 0) + 1
        top_colors = [c for c,_ in sorted(color_counts.items(), key=lambda x: x[1], reverse=True)[:2]]
        if not top_colors:
            top_colors = ["gray"]

        desc[name] = {
            "body": rng.choice(TEMPLATE_BODIES),
            "colors": ", ".join(top_colors),
            "mood": rng.choice(TEMPLATE_MOODS),
            "extras": rng.choice(TEMPLATE_EXTRAS),
        }
    return desc

# build and optionally apply overrides
auto_desc = build_pokedesc(CONCEPTS_ROOT)
if OVERRIDES_JSON and OVERRIDES_JSON.exists():
    with OVERRIDES_JSON.open("r", encoding="utf-8") as f:
        overrides = json.load(f)
    for k, v in overrides.items():
        auto_desc.setdefault(k, {}).update(v)

OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUT_JSON.write_text(json.dumps(auto_desc, indent=2), encoding="utf-8")
print(f"Wrote auto POKEDESC → {OUT_JSON}")



Wrote auto POKEDESC → results/pokemon-images/pokedesc_autofill.json


In [20]:
# Load pokemon descriptors
#POKEDESC = json.loads(Path("./results/pokemon-images/pokedesc_autofill.json").read_text())
print(auto_desc)



{'absol': {'body': 'fantasy mascot-like creature', 'colors': 'white, gray', 'mood': 'calm', 'extras': 'soft bounce light, gentle shadow on ground'}, 'aegislash-blade': {'body': 'creature with simplified anatomy, iconic silhouette', 'colors': 'white, gray', 'mood': 'vigilant', 'extras': 'soft bounce light, gentle shadow on ground'}, 'alakazam': {'body': 'compact creature, stylized proportions', 'colors': 'white, black', 'mood': 'mischievous', 'extras': 'dynamic pose, slight motion blur'}, 'amaura': {'body': 'fantasy mascot-like creature', 'colors': 'white, gray', 'mood': 'vigilant', 'extras': 'clean outline, subtle rim light'}, 'ampharos': {'body': 'fantasy mascot-like creature', 'colors': 'yellow, white', 'mood': 'stoic', 'extras': 'soft bounce light, gentle shadow on ground'}, 'annihilape': {'body': 'fantasy mascot-like creature', 'colors': 'black, white', 'mood': 'mischievous', 'extras': 'soft bounce light, gentle shadow on ground'}, 'arcanine': {'body': 'fantasy mascot-like creature

In [5]:
# # SD v1.5 — Hybrid: Global LoRA + Per‑Character Textual Inversion
# Cells TI‑1..TI‑3: add special tokens (one per Pokémon dir) and train token embeddings.
# Minimal, MPS‑safe, extendable to ~100 tokens.

# %%
# TI‑1 — Discover concepts, add tokens, init embeddings
from __future__ import annotations
from pathlib import Path
from typing import List, Dict
import json
import torch

import torch
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from diffusers import StableDiffusionPipeline
from transformers import CLIPTokenizer, CLIPTextModel
from local_tools import get_device

DEVICE = get_device()
MODEL_ID = "runwayml/stable-diffusion-v1-5"

vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae")
text_tok = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_enc = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder")
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")
scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")


vae.to(DEVICE).eval()
text_enc.to(DEVICE).eval()
unet.to(DEVICE).train(False) # will re‑enable grads only for LoRA in Cell 3


# Config — point to the parent directory containing one subfolder per Pokémon
CONCEPTS_ROOT = Path("./datasets/pokemon-images")  # e.g., ./datasets/pokemon-images/gengar, ./.../pikachu, ...
TOKEN_PREFIX = "<pk_"     # tokens become like <pk_gengar>
TOKEN_SUFFIX = ">"
CLASS_WORD   = "pokemon"  # keeps style anchor consistent


# Discover concept folders
concept_dirs = sorted([p for p in CONCEPTS_ROOT.iterdir() if p.is_dir()])
concept_names = [p.name for p in concept_dirs]
print(f"Found {len(concept_names)} concepts:", concept_names[:10], ("..." if len(concept_names)>10 else ""))

# Build tokens
concept_to_token: Dict[str, str] = {name: f"{TOKEN_PREFIX}{name}{TOKEN_SUFFIX}" for name in concept_names}
SPECIAL_TOKENS: List[str] = list(concept_to_token.values())

# Add tokens to tokenizer (idempotent)
added = text_tok.add_tokens([t for t in SPECIAL_TOKENS if t not in text_tok.get_vocab()])
if added:
    text_enc.to('cpu')
    text_enc.resize_token_embeddings(len(text_tok))
    text_enc.to(DEVICE)
print(f"Added {added} new tokens. Vocab size: {len(text_tok)}")

# Init each new token embedding from the class word vector (+ tiny noise)
base_id = text_tok.convert_tokens_to_ids(CLASS_WORD)
emb = text_enc.get_input_embeddings()
with torch.no_grad():
    base_vec = emb.weight[base_id].detach().clone()
    for name, tok in concept_to_token.items():
        tid = text_tok.convert_tokens_to_ids(tok)
        if tid is None or tid < 0:
            raise RuntimeError(f"Token not found after add_tokens: {tok}")
        # Only initialize if it looks fresh (heuristic: norm small)
        if emb.weight[tid].norm().item() < 1e-6 or True:
            emb.weight[tid].copy_(base_vec + 0.01 * torch.randn_like(base_vec))

# Persist mapping for later (inference, bookkeeping)
MAP_PATH = CONCEPTS_ROOT / "_ti_tokens.json"
MAP_PATH.write_text(json.dumps(concept_to_token, indent=2), encoding="utf-8")
print("Saved token map →", MAP_PATH)

# %%
# TI‑2 — Dataset for a single concept (auto‑caption with its trigger token)
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

class TIConceptDataset(Dataset):
    def __init__(self, folder: Path, token: str, size: int = 512, center_crop: bool = True,
                 cache_latents: bool = True, vae: AutoencoderKL | None = None, device: torch.device | None = None,
                 class_word: str = CLASS_WORD):
        self.paths = sorted([p for p in folder.glob("**/*") if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}])
        if not self.paths:
            raise FileNotFoundError(f"No images under {folder}")
        self.caption = f"{token} {class_word}"
        self.cache_latents = cache_latents
        self.vae = vae
        self.device = device
        self.tf = T.Compose([
            T.Resize(size, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(size) if center_crop else T.RandomCrop(size),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.latents: Dict[int, torch.Tensor] = {}

    def __len__(self): return len(self.paths)

    @torch.no_grad()
    def _encode_latent(self, img: torch.Tensor) -> torch.Tensor:
        assert self.vae is not None and self.device is not None
        img = img.unsqueeze(0).to(self.device)
        lat = self.vae.encode(img).latent_dist.sample() * 0.18215
        return lat.squeeze(0).cpu()

    def __getitem__(self, i: int):
        img = Image.open(self.paths[i]).convert("RGB")
        px = self.tf(img)
        if self.cache_latents:
            if i not in self.latents:
                self.latents[i] = self._encode_latent(px)
            return {"latent": self.latents[i], "caption": self.caption}
        return {"pixel_values": px, "caption": self.caption}


def build_loader_mps_safe(ds: Dataset, batch_size: int = 2) -> DataLoader:
    num_workers = 0 if DEVICE.type in ("mps","cpu") else 4
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        prefetch_factor=None if num_workers==0 else 2,
        persistent_workers=False if num_workers==0 else True,
        pin_memory=False,
        drop_last=True,
    )

print("TI dataset scaffolding ready.")

# %%
# TI‑3 — Train token embeddings (one concept at a time). UNet stays LoRA‑ready but frozen.
from typing import Optional

@torch.no_grad()
def _tok_ids(caps: List[str]) -> torch.Tensor:
    return text_tok(caps, padding="max_length", truncation=True, max_length=77, return_tensors="pt").input_ids.to(DEVICE)


def train_one_token(concept_name: str, steps: int = 1000, batch_size: int = 2, lr: float = 5e-3, cache_latents: bool = True) -> None:
    token = concept_to_token[concept_name]
    folder = CONCEPTS_ROOT / concept_name
    ds = TIConceptDataset(folder, token, size=512, center_crop=True,
                          cache_latents=cache_latents, vae=vae if cache_latents else None, device=DEVICE)
    loader = build_loader_mps_safe(ds, batch_size=batch_size)

    # freeze everything except the embedding matrix
    vae.eval(); unet.eval()
    for p in unet.parameters(): p.requires_grad_(False)
    text_enc.train(True)
    for p in text_enc.parameters(): p.requires_grad_(False)

    # target embedding row
    emb_mod = text_enc.get_input_embeddings()
    emb = emb_mod.weight
    emb.requires_grad_(True)  # 🔑 ensure grad required every call

    tid = text_tok.convert_tokens_to_ids(token)
    assert isinstance(tid, int) and 0 <= tid < emb.size(0), f"Bad token id for {token}: {tid}"
    mask = torch.zeros_like(emb, dtype=torch.bool); mask[tid] = True

    opt = torch.optim.AdamW([emb], lr=lr, weight_decay=0.0)

    print(f"Training token {token} for {steps} steps on {len(ds)} imgs…")
    step = 0
    while step < steps:
        for batch in loader:
            if step >= steps: break

            latents = (batch.get("latent") if cache_latents else
                       (vae.encode(batch["pixel_values"].to(DEVICE)).latent_dist.sample() * 0.18215)).to(DEVICE)
            noise = torch.randn_like(latents)
            t = torch.randint(0, scheduler.config.num_train_timesteps, (latents.size(0),), device=DEVICE, dtype=torch.long)
            noisy = scheduler.add_noise(latents, noise, t)

            caps = batch["caption"] if isinstance(batch["caption"], list) else [batch["caption"]]
            ids = text_tok(caps, padding="max_length", truncation=True, max_length=77, return_tensors="pt").input_ids.to(DEVICE)
            ctx = text_enc(ids).last_hidden_state  # grads flow into emb

            pred = unet(noisy, t, encoder_hidden_states=ctx).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())

            opt.zero_grad(set_to_none=True)
            loss.backward()

            # 🔒 mask grads to the single row (no register_hook needed)
            if emb.grad is not None:
                emb.grad[~mask] = 0

            torch.nn.utils.clip_grad_norm_([emb], 1.0)
            opt.step()

            step += 1
            if step % 50 == 0:
                print(f"{concept_name:>16s} | step {step:05d} | loss {loss.item():.4f}")

    # save vector
    vec = emb.detach().cpu()[tid]
    out_dir = Path("./results/pokemon-images/ti_embeddings"); out_dir.mkdir(parents=True, exist_ok=True)
    torch.save({"token": token, "vector": vec}, out_dir / f"{concept_name}.pt")
    print(f"Saved TI vector → {out_dir/concept_name}.pt")


print("TI training helpers ready. Example usage:")
print("train_one_token('gengar', steps=1000, batch_size=2, lr=5e-3)")

# %%
# TI-4 — Batch trainer: loop all concepts and train TI embeddings
from time import perf_counter

def estimate_steps(n_images: int, steps_per_image: int = 40, *, min_steps: int = 600, max_steps: int = 2000) -> int:
    """Rough heuristic: more images → more steps; clamp to sane bounds."""
    return int(max(min_steps, min(max_steps, n_images * steps_per_image)))

def train_all_tokens(
    steps_per_image: int = 40,
    min_steps: int = 600,
    max_steps: int = 2000,
    batch_size: int = 2,
    lr: float = 5e-3,
    overwrite_existing: bool = False,
    only: list[str] | None = None,
):
    start = perf_counter()
    done = 0
    errors: list[tuple[str, str]] = []

    # Load (name -> token) mapping
    token_map_path = CONCEPTS_ROOT / "_ti_tokens.json"
    concept_to_token_local: Dict[str, str] = json.loads(token_map_path.read_text())

    targets = [p.name for p in concept_dirs]
    if only:
        targets = [t for t in targets if t in set(only)]

    out_dir = Path("./results/pokemon-images/ti_embeddings"); out_dir.mkdir(parents=True, exist_ok=True)

    for name in targets:
        vec_path = out_dir / f"{name}.pt"
        if vec_path.exists() and not overwrite_existing:
            print(f"[skip] {name}: {vec_path.name} exists")
            done += 1
            continue

        folder = CONCEPTS_ROOT / name
        n_imgs = len([p for p in folder.glob("**/*") if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}])
        if n_imgs == 0:
            print(f"[warn] {name}: no images; skipping")
            continue
        steps = estimate_steps(n_imgs, steps_per_image, min_steps=min_steps, max_steps=max_steps)
        print(f"[run ] {name}: {n_imgs} imgs → {steps} steps")
        try:
            train_one_token(name, steps=steps, batch_size=batch_size, lr=lr, cache_latents=True)
            done += 1
        except Exception as e:
            errors.append((name, str(e)))
            print(f"[fail] {name}: {e}")

    dt = perf_counter() - start
    print(f"Batch TI done: {done}/{len(targets)} ok in {dt/60:.1f} min")
    if errors:
        print("Errors:")
        for n, msg in errors:
            print(" -", n, "→", msg)

print("Batch trainer ready. Example:")
print("train_all_tokens(steps_per_image=40, min_steps=800, max_steps=1600, batch_size=2, lr=5e-3)")




Found 106 concepts: ['absol', 'aegislash-blade', 'alakazam', 'amaura', 'ampharos', 'annihilape', 'arcanine', 'arceus', 'articuno', 'bayleef'] ...


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Added 106 new tokens. Vocab size: 49514
Saved token map → datasets/pokemon-images/_ti_tokens.json
TI dataset scaffolding ready.
TI training helpers ready. Example usage:
train_one_token('gengar', steps=1000, batch_size=2, lr=5e-3)
Batch trainer ready. Example:
train_all_tokens(steps_per_image=40, min_steps=800, max_steps=1600, batch_size=2, lr=5e-3)


In [6]:
# Train all remaining tokens; skips ones already saved in ti_embeddings/
train_all_tokens(
    steps_per_image=40,   # heuristic; bump to 50–70 for small folders, lower for big
    min_steps=800,        # floor
    max_steps=1600,       # cap
    batch_size=2,         # MPS-safe; raise only if you see headroom
    lr=5e-3,              # standard for TI
    overwrite_existing=OVERWRITE_EXISTING_TOKENS,
)


[skip] absol: absol.pt exists
[skip] aegislash-blade: aegislash-blade.pt exists
[skip] alakazam: alakazam.pt exists
[skip] amaura: amaura.pt exists
[skip] ampharos: ampharos.pt exists
[skip] annihilape: annihilape.pt exists
[skip] arcanine: arcanine.pt exists
[skip] arceus: arceus.pt exists
[skip] articuno: articuno.pt exists
[skip] bayleef: bayleef.pt exists
[skip] bewear: bewear.pt exists
[skip] bisharp: bisharp.pt exists
[skip] blaziken: blaziken.pt exists
[skip] bulbasaur: bulbasaur.pt exists
[skip] chandelure: chandelure.pt exists
[skip] charizard: charizard.pt exists
[skip] cinderace: cinderace.pt exists
[skip] crobat: crobat.pt exists
[skip] darkrai: darkrai.pt exists
[skip] deoxys-normal: deoxys-normal.pt exists
[skip] ditto: ditto.pt exists
[skip] dragapult: dragapult.pt exists
[skip] dragonite: dragonite.pt exists
[skip] eevee: eevee.pt exists
[skip] electivire: electivire.pt exists
[skip] excadrill: excadrill.pt exists
[skip] flygon: flygon.pt exists
[skip] froslass: froslas

In [7]:

# %%
# TI-5 — Verify a single token embedding (e.g., Absol) via text→image
from pathlib import Path
import json, torch
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
from peft import PeftModel

VERIFY_NAME = "alakazam"                              # change to any trained concept name
ADAPTER_DIR = Path("./results/pokemon-images/lora_peft_minimal")  # set to None to skip global LoRA
VECTORS_DIR = Path("./results/pokemon-images/ti_embeddings")
TOKEN_MAP = Path("./datasets/pokemon-images/_ti_tokens.json")
OUT_DIR = Path("./results/pokemon-images/samples"); OUT_DIR.mkdir(parents=True, exist_ok=True)


# 1) Load token map and vector
concept_to_token = json.loads(TOKEN_MAP.read_text())
assert VERIFY_NAME in concept_to_token, f"{VERIFY_NAME} not in token map"
TRIGGER = concept_to_token[VERIFY_NAME]
vec_obj = torch.load(VECTORS_DIR / f"{VERIFY_NAME}.pt", map_location="cpu")
assert vec_obj["token"] == TRIGGER
vec = vec_obj["vector"].float()

# 2) Build base pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,   # MPS-friendly
    safety_checker=None,
)
# Simpler scheduler avoids heavy linalg on MPS
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.to(DEVICE)

# 3) (Optional) Load global style LoRA and MERGE (avoids wrapper forward issues)
if ADAPTER_DIR is not None and ADAPTER_DIR.exists():
    peft_wrapper = PeftModel.from_pretrained(pipe.unet, str(ADAPTER_DIR))
    pipe.unet = peft_wrapper.merge_and_unload().to(DEVICE)

# 4) Ensure trigger token exists, resize TE on CPU, and inject vector
added = 0
if TRIGGER not in pipe.tokenizer.get_vocab():
    added = pipe.tokenizer.add_tokens([TRIGGER])
if added:
    te_cpu = pipe.text_encoder.to("cpu")
    new_size = pipe.tokenizer.vocab_size + len(pipe.tokenizer.get_added_vocab())
    te_cpu.resize_token_embeddings(new_size)
    pipe.text_encoder = te_cpu.to(DEVICE)

tid = pipe.tokenizer.convert_tokens_to_ids(TRIGGER)
emb = pipe.text_encoder.get_input_embeddings()
with torch.no_grad():
    emb.weight[tid].copy_(vec.to(emb.weight.device))

# 5) Generate
prompt = f"{TRIGGER} pokemon, 3d render, studio lighting, high detail"
negative = "low quality, blurry, deformed, worst quality"

num_inference_steps = 50
guidance_scale = 7.5
seed = 123

if DEVICE.type == "mps":
    torch.manual_seed(seed); generator = None
else:
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

with torch.no_grad():
    out = pipe(
        prompt=prompt,
        negative_prompt=negative,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    )
img = out.images[0]
fp = OUT_DIR / f"verify_{VERIFY_NAME}.png"
img.save(fp)
print("saved →", fp)

/var/folders/2d/nd_kkrs166v10fz9h75cg5v00000gn/T/ipykernel_1275/1906650030.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vec_obj = torch.load(VECTORS_DIR / f"{VERIFY_

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  0%|          | 0/50 [00:00<?, ?it/s]

saved → results/pokemon-images/samples/verify_alakazam.png


In [19]:
POKEDESC['snorlax']

{'body': 'compact creature, stylized proportions',
 'colors': 'white, gray',
 'mood': 'vigilant',
 'extras': 'soft bounce light, gentle shadow on ground'}

In [ ]:

# ---- Helpers

def build_prompt(trigger: str, name: str, style: str = "3d", extra_tags: Optional[List[str]] = None) -> tuple[str, str]:
    d = POKEDESC.get(name, {})
    pos = [
        f"{trigger} pokemon",
        d.get("body", ""),
        d.get("colors", ""),
        d.get("mood", ""),
        d.get("extras", ""),
        STYLE_PRESETS[style]["positive"],
    ]
    if extra_tags:
        pos += extra_tags
    positive = ", ".join(filter(None, pos))
    negative = STYLE_PRESETS[style]["negative"]
    return positive, negative


def ensure_token_and_inject_vector(pipe: StableDiffusionPipeline, trigger: str, vec: torch.Tensor) -> None:
    # make sure token exists
    added = 0
    if trigger not in pipe.tokenizer.get_vocab():
        added = pipe.tokenizer.add_tokens([trigger])
    if added:
        te_cpu = pipe.text_encoder.to("cpu")
        new_size = pipe.tokenizer.vocab_size + len(pipe.tokenizer.get_added_vocab())
        te_cpu.resize_token_embeddings(new_size)
        pipe.text_encoder = te_cpu.to(DEVICE)
    tid = pipe.tokenizer.convert_tokens_to_ids(trigger)
    emb = pipe.text_encoder.get_input_embeddings()
    with torch.no_grad():
        emb.weight[tid].copy_(vec.to(emb.weight.device))


def sample_tokens(names: List[str], style: str = "3d", steps: int = 30, guidance: float = 7.5, seed: int = 123) -> Dict[str, Path]:
    # load maps/vectors
    tokmap = json.loads(TOKEN_MAP_PATH.read_text())

    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32, safety_checker=None
    )
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.to(DEVICE)

    # (optional) merge LoRA
    if ADAPTER_DIR is not None and ADAPTER_DIR.exists():
        peft_wrapper = PeftModel.from_pretrained(pipe.unet, str(ADAPTER_DIR))
        pipe.unet = peft_wrapper.merge_and_unload().to(DEVICE)

    # generator
    generator = None if DEVICE.type == "mps" else torch.Generator(device=DEVICE).manual_seed(seed)

    saved: Dict[str, Path] = {}
    for name in names:
        trigger = tokmap[name]
        vec_obj = torch.load(VECTORS_DIR / f"{name}.pt", map_location="cpu")
        vec = vec_obj["vector"].float()
        ensure_token_and_inject_vector(pipe, trigger, vec)
        pos, neg = build_prompt(trigger, name, style=style)
        with torch.no_grad():
            out = pipe(
                prompt=pos,
                negative_prompt=neg,
                num_inference_steps=steps,
                guidance_scale=guidance,
                generator=generator,
            )
        img = out.images[0]
        fp = OUT_DIR / f"{name}_{style}_s{seed}.png"
        img.save(fp)
        saved[name] = fp
        print("saved →", fp)
    return saved

# Example usage:
# saved = sample_tokens(["absol", "gengar", "pikachu"], style="card_art", steps=28, guidance=6.5, seed=7)